# POLSCI 3

## Week 7 Section Activity: Are AC Transit Routes Busier on Weekdays?

This activity covers the *p*-value ideas from Week 6 Lecture 2 and Week 7 Lecture 1: imagining a skeptic (Sarah), null hypotheses, and what a *p*-value does and does not tell us.

- It is intentionally harder than the in-class notebooks on purpose. Each question tells you *what* to calculate but you have to come up with the steps. Use your lecture notebooks and the R cheat sheet, and feel free to talk to others or ask me for help.
- After each question, you can run the **check cell** to see if you are right. For multiple choice, save your letter inside the quotes.
- This activity is not graded and for the multiple choice you could just put in every possible letter to check but this is for your own learning.
- The data is **real**: average daily ridership on AC Transit bus routes.
- Cells marked **(given)** have simulation code like the code in lecture. You don't need to be able to write it, but make sure you understand what it is doing.

In [ ]:
library(estimatr)
library(ggplot2)

ac <- read.csv('ps3_actransit_ridership.csv')
head(ac)

### Codebook

Each row is **one AC Transit route, in one month, on one day type**. The data covers 2023 to 2025.

- `route`: the bus route
- `type`: kind of route (`Local`, `Supplemental`, `M-Transbay`, `All Nighter`, `Transbay`)
- `year`, `month`: when the ridership was measured
- `day_type`: `"Weekday"`, `"Saturday"`, or `"Sunday"`
- `avg_daily_riders`: the average number of riders on that route per day, in that month, on that day type
- `all_three_days`: 1 if the route has ridership for **all three** day types in that month, 0 if not (some routes don't run on weekends, so this lets you compare the same routes across day types)

---
# Part A: Weekdays vs. Sundays

Your friend Sarah is a skeptic. She says that **the day of the week doesn't matter**: AC Transit routes are about equally busy on weekdays, Saturdays, and Sundays. You think she's wrong.

- **Null hypothesis (what Sarah believes):** day type has no relationship with how many riders a route gets.
- **Alternative hypothesis (what you want to show):** it does.

For Part A, use **October 2024** and only the routes that have data for all three day types (`all_three_days == 1`), so each route shows up once for each day type.

**Q1.** Estimate how many more average daily riders these routes have on **weekdays than on Sundays**. Save the full `difference_in_means()` output (weekday minus Sunday) as `q1.dim`.

In [ ]:
# Your code here


q1.dim <- NULL

In [ ]:
. = ottr::check("tests/q1.R")

**Q2.** What is the **null hypothesis** for the weekday vs. Sunday comparison?

- **A.** Routes have more riders on weekdays than on Sundays.
- **B.** AC Transit carries no riders on Sundays.
- **C.** Day type has no relationship with ridership: routes are equally busy on weekdays and Sundays on average.
- **D.** The gap you found in Q1 is exactly the true gap.

Save your choice as a letter (text in quotes) in `q2.answer`.

In [ ]:
q2.answer <- ""  # replace with your letter

In [ ]:
. = ottr::check("tests/q2.R")

### (given) What could happen by chance if day type doesn't matter?

If Sarah is right, it doesn't matter which rows are labeled Weekday, Saturday, or Sunday, so we can **shuffle** the day-type labels and see how big a gap luck alone produces. This cell does that 10,000 times, like Week 7 Lecture 1, and saves:

- `shuffled.wk.sun`: the 10,000 shuffled weekday-minus-Sunday gaps
- `shuffled.sat.sun`: the 10,000 shuffled Saturday-minus-Sunday gaps
- `actual.wk.sun` and `actual.sat.sun`: the gaps we actually see in the data

(This cell makes its own copy of the data, so it works no matter what you named your objects. It computes the gaps with `mean()`, so it runs fast. Also, `difference_in_means()` prints its own *p*-value in the `Pr(>|t|)` column. It's calculated differently than the one here, so don't expect them to match.)

In [ ]:
sim.data <- read.csv('ps3_actransit_ridership.csv')
sim.data <- subset(sim.data, year == 2024 & month == 'Oct' & all_three_days == 1)

# Function to re-randomize, or "shuffle", the day type
re.randomize <- function(input.data) {
    input.data$day_type <- sample(input.data$day_type)
    return(input.data)
}

# Function to compute the gap in average daily riders between two day types
compute.gap <- function(input.data, group1, group2) {
    mean(input.data$avg_daily_riders[input.data$day_type == group1]) -
        mean(input.data$avg_daily_riders[input.data$day_type == group2])
}

set.seed(11)
shuffled.wk.sun <- replicate(10000, compute.gap(re.randomize(sim.data), 'Weekday', 'Sunday'))
shuffled.sat.sun <- replicate(10000, compute.gap(re.randomize(sim.data), 'Saturday', 'Sunday'))

actual.wk.sun <- compute.gap(sim.data, 'Weekday', 'Sunday')
actual.sat.sun <- compute.gap(sim.data, 'Saturday', 'Sunday')

The red lines are the gaps we actually see. First weekday vs. Sunday, then Saturday vs. Sunday:

In [ ]:
qplot(shuffled.wk.sun, binwidth = 50) + geom_vline(xintercept = actual.wk.sun, color = 'red')

In [ ]:
qplot(shuffled.sat.sun, binwidth = 50) + geom_vline(xintercept = actual.sat.sun, color = 'red')

**Q3.** Find the *p*-value for weekday vs. Sunday: the proportion of shuffled gaps that are **at least as large** as the actual gap. Save it as `q3.p`.

In [ ]:
# Your code here


q3.p <- NULL

In [ ]:
. = ottr::check("tests/q3.R")

**Q4.** Do the same for Saturday vs. Sunday. Save the *p*-value as `q4.p`.

In [ ]:
# Your code here


q4.p <- NULL

In [ ]:
. = ottr::check("tests/q4.R")

**Q5.** Your *p*-value for weekday vs. Sunday (Q3) is below 0.05. Which statement is correct?

- **A.** There is less than a 5% chance that day type is unrelated to ridership.
- **B.** If day type were unrelated to ridership, a gap this large would show up in less than 5% of shuffles, so by convention we reject the null.
- **C.** It proves that routes are busier on weekdays because people commute.
- **D.** The p-value is the size of the gap, in riders.

Save your choice as a letter (text in quotes) in `q5.answer`.

In [ ]:
q5.answer <- ""  # replace with your letter

In [ ]:
. = ottr::check("tests/q5.R")

**Q6.** In Q4, the Saturday-minus-Sunday gap is positive (about 150 more riders per route on Saturdays), but the *p*-value is large (about 0.36). What is the best conclusion?

- **A.** Saturdays and Sundays are equally busy.
- **B.** Sundays are busier than Saturdays, because the p-value is large.
- **C.** The estimate must be wrong, because the p-value is above 0.05.
- **D.** The result is inconclusive: we can't rule out chance, though our best guess is that Saturdays are a bit busier.

Save your choice as a letter (text in quotes) in `q6.answer`.

In [ ]:
q6.answer <- ""  # replace with your letter

In [ ]:
. = ottr::check("tests/q6.R")

---
# Part B: Is 0.05 a magic line?

Sarah says: "Fine, weekdays beat Sundays. But what about **weekdays vs. Saturdays**?" You decide to check the same October comparison in three different years.

### (given) Weekday vs. Saturday in October 2023, 2024, and 2025

This cell does the same shuffle test for the **weekday-minus-Saturday** gap in each of the three Octobers (10,000 shuffles each). It saves the shuffled gaps (`shuffled.gap.2023`, `shuffled.gap.2024`, `shuffled.gap.2025`) and the actual gaps (`actual.gap.2023`, `actual.gap.2024`, `actual.gap.2025`). It again makes its own copy of the data.

In [ ]:
re.randomize <- function(input.data) {
    input.data$day_type <- sample(input.data$day_type)
    return(input.data)
}

compute.gap <- function(input.data, group1, group2) {
    mean(input.data$avg_daily_riders[input.data$day_type == group1]) -
        mean(input.data$avg_daily_riders[input.data$day_type == group2])
}

shuffle.weekday.saturday <- function(yr) {
    d <- read.csv('ps3_actransit_ridership.csv')
    d <- subset(d, year == yr & month == 'Oct' & all_three_days == 1 & day_type != 'Sunday')
    shuffled <- replicate(10000, compute.gap(re.randomize(d), 'Weekday', 'Saturday'))
    list(shuffled = shuffled, actual = compute.gap(d, 'Weekday', 'Saturday'))
}

set.seed(2023)
res.2023 <- shuffle.weekday.saturday(2023)
res.2024 <- shuffle.weekday.saturday(2024)
res.2025 <- shuffle.weekday.saturday(2025)

shuffled.gap.2023 <- res.2023$shuffled
shuffled.gap.2024 <- res.2024$shuffled
shuffled.gap.2025 <- res.2025$shuffled

actual.gap.2023 <- res.2023$actual
actual.gap.2024 <- res.2024$actual
actual.gap.2025 <- res.2025$actual

In [ ]:
# the actual weekday-minus-Saturday gaps (riders per route)
c(actual.gap.2023, actual.gap.2024, actual.gap.2025)

**Q7.** Find the *p*-value for weekday vs. Saturday in each October: the proportion of shuffled gaps at least as large as the actual gap. Save them as `q7.2023`, `q7.2024`, and `q7.2025`.

In [ ]:
# Your code here


q7.2023 <- NULL
q7.2024 <- NULL
q7.2025 <- NULL

In [ ]:
. = ottr::check("tests/q7.R")

**Q8.** The weekday-minus-Saturday gaps were similar in all three years (roughly 680 to 820 more riders per route on weekdays). A writer says: *"Weekdays were significantly busier than Saturdays in 2024 and 2025, but in 2023 there was no difference."* What is the best response?

- **A.** The 2023 evidence is nearly as strong as in the other years. 0.05 is an arbitrary line, so saying there was "no difference" in 2023 overstates the contrast.
- **B.** That's correct: 2023 isn't significant, so there was no difference.
- **C.** p-values below 0.05 mean the gap was exactly the same size in 2024 and 2025.
- **D.** 2023 must be biased, because its p-value is higher.

Save your choice as a letter (text in quotes) in `q8.answer`.

In [ ]:
q8.answer <- ""  # replace with your letter

In [ ]:
. = ottr::check("tests/q8.R")

---
# Part C: What if there is nothing to find?

Sarah has one last challenge: *"If you test enough things, you'll always find something significant."* Let's see if she's right.

Take the **weekday ridership of every AC Transit route in October 2024** (real data). Imagine **20 student groups**, and each one randomly splits the routes into "Group A" and "Group B", then tests whether Group A has more riders than Group B. The labels mean nothing, so **the null hypothesis is true for every group**.

### (given) The 20 groups

This cell runs the 20 groups' shuffle tests (1,000 shuffles each) and saves each group's *p*-value in `placebo.p`. Once again it makes its own copy of the data.

In [ ]:
placebo.data <- read.csv('ps3_actransit_ridership.csv')
placebo.data <- subset(placebo.data, year == 2024 & month == 'Oct' & day_type == 'Weekday')
riders <- placebo.data$avg_daily_riders

set.seed(1)
placebo.p <- replicate(20, {
    random.group <- sample(rep(c('A', 'B'), length.out = length(riders)))     # a meaningless random split
    actual <- mean(riders[random.group == 'A']) - mean(riders[random.group == 'B'])
    shuffled <- replicate(1000, {
        g <- sample(random.group)
        mean(riders[g == 'A']) - mean(riders[g == 'B'])
    })
    mean(shuffled >= actual)
})

round(placebo.p, 3)

**Q9.** How many of the 20 groups got a *p*-value below 0.05? Save the count as `q9.n`.

In [ ]:
# Your code here


q9.n <- NULL

In [ ]:
. = ottr::check("tests/q9.R")

**Q10.** The labels were meaningless, yet some groups may have gotten a *p*-value below 0.05. Why?

- **A.** The routes in Group A really are busier than the routes in Group B.
- **B.** The shuffle test is broken.
- **C.** A p-value below 0.05 happens about 5% of the time (1 in 20) even when the null is true, just by chance.
- **D.** Real data always has a few significant results.

Save your choice as a letter (text in quotes) in `q10.answer`.

In [ ]:
q10.answer <- ""  # replace with your letter

In [ ]:
. = ottr::check("tests/q10.R")

**Q11.** Sarah says: *"Your weekday vs. Sunday p-value proves that weekdays cause higher ridership."* What is the best response?

- **A.** Agree: a p-value below 0.05 always proves a causal effect.
- **B.** Agree: bigger gaps always mean causation.
- **C.** Disagree: a p-value can never be below 0.05 with real data.
- **D.** Disagree: the p-value only says a gap this large would be unlikely from luck alone if day type were unrelated to ridership. It doesn't say why, and weekday and weekend riders (for example, commuters) are different people riding for different reasons.

Save your choice as a letter (text in quotes) in `q11.answer`.

In [ ]:
q11.answer <- ""  # replace with your letter

In [ ]:
. = ottr::check("tests/q11.R")

---
## Done!

If you finish early, try the optional `dplyr` notebook.